# 全链长，bb分析

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go

def interactive_visualize(
    coords_clean: torch.Tensor,
    coords_noisy: torch.Tensor,
    atom_idx: int = 1,  # CA  # 1
    output_html: str = None
):
    """Plotly 交互式 3D 图，可旋转缩放，适合调试"""
    clean_ca = coords_clean[:, atom_idx].cpu().numpy()
    noisy_ca = coords_noisy[:, atom_idx].cpu().numpy()
    displacement = np.linalg.norm(clean_ca - noisy_ca, axis=1)
    
    fig = go.Figure()
    
    # 原始结构
    fig.add_trace(go.Scatter3d(
        x=clean_ca[:, 0], y=clean_ca[:, 1], z=clean_ca[:, 2],
        mode='markers',
        name='Clean',
        line=dict(width=2, color='blue'),
        marker=dict(size=3, color='blue', opacity=0.7)
    ))
    
    # 加噪结构（用颜色编码位移大小）
    fig.add_trace(go.Scatter3d(
        x=noisy_ca[:, 0], y=noisy_ca[:, 1], z=noisy_ca[:, 2],
        # mode='lines+markers',
        # name='Noisy',
        # marker=dict(
        #     size=5,
        #     color=displacement,
        #     colorscale='RdYlBu_r',  # 红=大扰动，蓝=小扰动
        #     colorbar=dict(title="Displacement (Å)"),
        #     opacity=0.8
        # )
        mode='markers',
        # name='Noisy',
        line=dict(width=1, color='red'),
        marker=dict(size=2, color='red', opacity=0.7)
    ))
    
    fig.update_layout(
        title="Clean vs Noisy (Color = Displacement)",
        scene=dict(
            xaxis_title='X (Å)',
            yaxis_title='Y (Å)',
            zaxis_title='Z (Å)',
            aspectmode='data'  # 保持 1:1:1 比例
        ),
        width=1000,
        height=800
    )
    
    if output_html:
        fig.write_html(output_html)
        print(f"✅ Interactive plot saved: {output_html}")
    
    fig.show()
    # return fig


# save
"""

# 预测对比原始数据
torch.save({
    'perturb': inputs['cord-p'],
    'pre': pred,
    'clean': tgt
}, '/root/private_data/luog/codex/IgGM/see/seefile/perturb_per_clean.pt')


"""


# 加载文件
loaded_dict = torch.load(
    '/root/private_data/luog/codex/IgGM2/see/seefile/S28_1780929112.pt',
    map_location='cpu',    
    weights_only=True       
)

# 使用


# interactive_visualize(
#     coords_clean = loaded_dict['clean'].squeeze(0).detach()[20:35,:,:] ,
#     coords_noisy = loaded_dict['perturb'].squeeze(0).detach()[20:35,:,:] ,
# )
# interactive_visualize(
#     coords_clean = loaded_dict['perturb'].squeeze(0).detach() ,
#     coords_noisy = loaded_dict['pre'].squeeze(0).detach() ,
# )

interactive_visualize(
    coords_clean = loaded_dict['clean'].squeeze(0).detach()[20:35,:,:] ,
    coords_noisy = loaded_dict['pre'].squeeze(0).detach()[20:35,:,:] ,
)

interactive_visualize(
    coords_clean = loaded_dict['clean'].squeeze(0).detach()[50:70,:,:] ,
    coords_noisy = loaded_dict['pre'].squeeze(0).detach()[50:70,:,:] ,
)


# interactive_visualize(
#     coords_clean = loaded_dict['clean'].squeeze(0).detach() ,
#     coords_noisy = loaded_dict['perturb'].squeeze(0).detach(),
# )
# interactive_visualize(
#     coords_clean = loaded_dict['clean'].squeeze(0).detach(),
#     coords_noisy = loaded_dict['pre'].squeeze(0).detach(),
# )



# 逐个氨基酸分析

In [ ]:
# # loaded_dict['clean'].squeeze(0).detach().shape
# # # a ="QDQLQQSGAELVRPGASVKLSCKALGYIFTDYEIHWVKQTPVHGLEWIGGIHPGSSGTAYNQKFKGKATLTADKSSTTAFMELSSLTSEDSAVYYCTRKDYWGQGTLVTVSAAKTTAPSVYPLVPVCGGTTGSSVTLGCLVKGYFPEPVTLTWNSGSLSSGVHTFPALLQSGLYTLSSSVTVTSNTWPSQTITCNVAHPASSTKVDKKIEPRV"
# # # a[25]
# # H = "QDQLQQSGAELVRPGASVKLSCKALGYIFTDYEIHWVKQTPVHGLEWIGGIHPGSSGTAYNQKFKGKATLTADKSSTTAFMELSSLTSEDSAVYYCTRKDYWGQGTLVTVSAAKTTAPSVYPLVPVCGGTTGSSVTLGCLVKGYFPEPVTLTWNSGSLSSGVHTFPALLQSGLYTLSSSVTVTSNTWPSQTITCNVAHPASSTKVDKKIEPRV"
# # L="DIKMTQSPSSMYTSLGERVTITCKASQDINSFLTWFLQKPGKSPKTLIYRANRLMIGVPSRFSGSGSGQTYSLTISSLEYEDMGIYYCLQYDDFPLTFGAGTKLDLKRADAAPTVSIFPPSSEQLTSGTASVVCFLNNFYPKEINVKWKIDGSERQNGVLDSWTEQDSKDSTYSMSSTLTLTKDEYERHNSYTCEATHKTSTSPIVKSFNRNEC"
# # len(H),len(L)

# for i in range(0, 100):
#     if loaded_dict['clean'].squeeze(0).detach()[213+i,-1,-1]+183 >20:
#         print(i)

In [2]:
import torch
import numpy as np
import plotly.graph_objects as go

# 【新增函数：计算并打印虚拟原子到 N 和 O 的距离】
def print_virtual_atom_distances(coords_noisy: torch.Tensor, res_idx: int):
    """计算预测结构中虚拟原子(4-13)到 N(0) 和 O(3) 的欧氏距离(埃)"""
    # 获取特定残基的坐标 (注意：这里直接计算原始预测坐标，不加 jitter)
    atoms = coords_noisy[res_idx].cpu().numpy()
    
    n_coord = atoms[0]
    o_coord = atoms[3]
    
    print(f"\n=== 残基 {res_idx} 预测结构(Noisy) 虚拟原子距离分析 ===")
    print(f"{'Atom':<8} | {'Dist to N (Å)':<15} | {'Dist to O (Å)':<15}")
    print("-" * 42)
    
    for i in range(4, 14):
        v_coord = atoms[i]
        dist_to_n = np.linalg.norm(v_coord - n_coord)
        dist_to_o = np.linalg.norm(v_coord - o_coord)
        print(f"Atom {i:<3} | {dist_to_n:<15.4f} | {dist_to_o:<15.4f}")
    print("==========================================\n")


def interactive_visualize(
    coords_clean: torch.Tensor,
    coords_noisy: torch.Tensor,
    res_idx: int = 0,  
    output_html: str = None
):
    """Plotly 交互式 3D 图，用于调试 atom14 表示法中虚拟原子的空间分布"""
    
    # 提取坐标时加上 .copy()，避免原地修改影响外部数据
    clean_atoms = coords_clean[res_idx].cpu().numpy().copy()
    noisy_atoms = coords_noisy[res_idx].cpu().numpy().copy()
    
    # # 【新增：添加虚伪的微小扰动】
    # # 给虚拟原子（索引 4~13）加上正态分布的随机位移，以便在 3D 图中散开
    # jitter_scale = 0.1  # 扰动大小 (Å)，如果还是看不清可以稍微调大
    # clean_atoms[4:14] += np.random.normal(scale=jitter_scale, size=(10, 3))
    # noisy_atoms[4:14] += np.random.normal(scale=jitter_scale, size=(10, 3))
    
    # 为14个原子生成悬停标签，方便你在网页上确认是哪个原子
    atom_labels = []
    for i in range(14):
        if i == 0: atom_labels.append("0: N")
        elif i == 1: atom_labels.append("1: CA")
        elif i == 2: atom_labels.append("2: C")
        elif i == 3: atom_labels.append("3: O")
        else: atom_labels.append(f"{i}")
        # else: atom_labels.append(f"{i}: Virtual/Side")

    
    atom_labels2 = []
    for i in range(14):
        if i == 0: atom_labels2.append("0: -N")
        elif i == 1: atom_labels2.append("1: -CA")
        elif i == 2: atom_labels2.append("2: -C")
        elif i == 3: atom_labels2.append("3: -O")
        else: atom_labels2.append(f"{i}-")

    # 颜色编码：N设为绿色，O设为橙色，CA/C设为灰色，虚拟原子设为红色
    marker_colors = ['green' if i==0 else 'orange' if i==3 else 'gray' if i in [1,2] else 'red' for i in range(14)]

    fig = go.Figure()
    
    # 原始/真实结构 (Clean) 
    fig.add_trace(go.Scatter3d(
        x=clean_atoms[:, 0], y=clean_atoms[:, 1], z=clean_atoms[:, 2],
        mode='markers+text',
        text=atom_labels,
        textposition="bottom center",
        name='Clean (Target)',
        marker=dict(size=6, color=marker_colors, symbol='circle-open', opacity=0.8) # 'blue'
    ))
    
    # 预测结构 (Predicted) 
    fig.add_trace(go.Scatter3d(
        x=noisy_atoms[:, 0], y=noisy_atoms[:, 1], z=noisy_atoms[:, 2],
        mode='markers+text',
        text=atom_labels2,
        textposition="top center",
        name='Predicted',
        marker=dict(size=5, color=marker_colors, opacity=0.9)
    ))
    
    fig.update_layout(
        title=f"Residue {res_idx} Debug: Green=N, Orange=O, Red=Virtual Atoms (with Jitter)",
        scene=dict(
            xaxis_title='X (Å)',
            yaxis_title='Y (Å)',
            zaxis_title='Z (Å)',
            aspectmode='data'  
        ),
        width=1000,
        height=800
    )
    
    if output_html:
        fig.write_html(output_html)
        print(f"✅ Interactive plot saved: {output_html}")
    
    fig.show()
    

# 加载文件
loaded_dict = torch.load(
    '/root/private_data/luog/codex/IgGM2/see/seefile/S28_1780859786.pt', 
    map_location='cpu',    
    weights_only=True       
)

# 提取你想要查看的数据片段
coords_clean_segment = loaded_dict['clean'].squeeze(0).detach()[:,:,:] # clean
coords_noisy_segment = loaded_dict['pre'].squeeze(0).detach()[:,:,:] # pre 
target_res_idx = 55

# 【新增调用：在画图前打印距离信息】
print_virtual_atom_distances(
    coords_noisy=coords_noisy_segment, 
    res_idx=target_res_idx
)

# 使用示例：检查残基的预测情况
interactive_visualize(
    coords_clean = coords_clean_segment,
    coords_noisy = coords_noisy_segment,
    res_idx = target_res_idx  
)


=== 残基 55 预测结构(Noisy) 虚拟原子距离分析 ===
Atom     | Dist to N (Å)   | Dist to O (Å)  
------------------------------------------
Atom 4   | 2.4650          | 1.7042         
Atom 5   | 3.3359          | 2.8292         
Atom 6   | 1.2568          | 3.3535         
Atom 7   | 0.5213          | 2.5715         
Atom 8   | 1.3019          | 3.7018         
Atom 9   | 0.9326          | 3.2806         
Atom 10  | 1.2678          | 3.6129         
Atom 11  | 1.1793          | 3.4761         
Atom 12  | 0.9858          | 3.1867         
Atom 13  | 0.8701          | 3.0701         



# loop对比查看

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go

def interactive_visualize_loops(
    coords_clean: torch.Tensor,
    coords_pred: torch.Tensor,
    atom_idx: int = 1,  # 默认 1 代表 CA 原子
    output_html: str = None
):
    """
    针对形状为 [batch, n_loops, max_res, 14, 3] 的张量进行交互式 3D 可视化。
    """
    # 1. 提取目标原子 (CA) 并移除 batch 维度，转换为 numpy
    # 提取后形状变为 (6, 11, 3)
    if coords_clean.dim() == 5:
        coords_clean = coords_clean.squeeze(0)
    if coords_pred.dim() == 5:
        coords_pred = coords_pred.squeeze(0)
        
    clean_ca = coords_clean[:, :, atom_idx, :].detach().cpu().numpy()
    pred_ca = coords_pred[:, :, atom_idx, :].detach().cpu().numpy()
    
    n_loops = clean_ca.shape[0]
    
    fig = go.Figure()
    
    # 颜色调色板 (可以让不同 loop 颜色有一点点区分度，但保持整体 Clean为蓝，Pred为红)
    clean_colors = ['#1f77b4', '#00008B', '#4169E1', '#1E90FF', '#00BFFF', '#87CEEB']
    pred_colors = ['#d62728', '#8B0000', '#B22222', '#FF4500', '#FF6347', '#FF7F50']
    loop_names = ['Loop 1 (e.g. H1)', 'Loop 2 (e.g. H2)', 'Loop 3 (e.g. H3)', 
                  'Loop 4 (e.g. L1)', 'Loop 5 (e.g. L2)', 'Loop 6 (e.g. L3)']

    for i in range(n_loops):
        loop_c = clean_ca[i]
        loop_p = pred_ca[i]
        
        # 2. 过滤 Padding (坐标全为0的点)
        # 计算每个残基到原点的距离，如果非常接近 0，则认为是 padding
        valid_mask_c = np.linalg.norm(loop_c, axis=-1) > 1e-3
        valid_mask_p = np.linalg.norm(loop_p, axis=-1) > 1e-3
        
        valid_loop_c = loop_c[valid_mask_c]
        valid_loop_p = loop_p[valid_mask_p]
        
        if len(valid_loop_c) == 0 or len(valid_loop_p) == 0:
            continue  # 如果这个 loop 全是 padding，跳过

        # 3. 绘制 Clean 结构 (蓝色系)
        fig.add_trace(go.Scatter3d(
            x=valid_loop_c[:, 0], y=valid_loop_c[:, 1], z=valid_loop_c[:, 2],
            mode='lines+markers',
            name=f'Clean - {loop_names[i]}',
            legendgroup='Clean',
            legendgrouptitle_text='Clean Target',
            line=dict(width=4, color=clean_colors[i % 6]),
            marker=dict(size=4, color=clean_colors[i % 6], opacity=0.8)
        ))
        
        # 4. 绘制 Pred/Noisy 结构 (红色系)
        fig.add_trace(go.Scatter3d(
            x=valid_loop_p[:, 0], y=valid_loop_p[:, 1], z=valid_loop_p[:, 2],
            mode='lines+markers',
            name=f'Pred - {loop_names[i]}',
            legendgroup='Predicted',
            legendgrouptitle_text='Predicted Output',
            line=dict(width=3, color=pred_colors[i % 6], dash='dash'), # 用虚线区分
            marker=dict(size=4, color=pred_colors[i % 6], opacity=0.8)
        ))

        # 5. (可选) 绘制 Clean 和 Pred 之间的位移连线
        # 只在长度相等时绘制位移连线（真实长度）
        if len(valid_loop_c) == len(valid_loop_p):
            for j in range(len(valid_loop_c)):
                fig.add_trace(go.Scatter3d(
                    x=[valid_loop_c[j, 0], valid_loop_p[j, 0]],
                    y=[valid_loop_c[j, 1], valid_loop_p[j, 1]],
                    z=[valid_loop_c[j, 2], valid_loop_p[j, 2]],
                    mode='lines',
                    showlegend=False,
                    legendgroup='Displacement',
                    hoverinfo='skip',
                    line=dict(width=1, color='gray', dash='dot')
                ))

    # 设置布局，保持三轴比例 1:1:1
    fig.update_layout(
        title="Interactive Loop Alignment (Clean vs Predicted)",
        scene=dict(
            xaxis_title='X (Å)',
            yaxis_title='Y (Å)',
            zaxis_title='Z (Å)',
            aspectmode='data'  # 非常重要：防止蛋白质被拉伸变形
        ),
        width=1200,
        height=900,
        legend=dict(groupclick="toggleitem") # 点击图例组名可以隐藏整个组
    )
    
    fig.show()
    return fig

loaded_dict = torch.load(
    '/root/private_data/luog/codex/IgGM2/see/seefile/S28_loop.pt',
    map_location='cpu',    
    weights_only=True       
)

interactive_visualize_loops(
    coords_clean=loaded_dict['clean_cdrblock'],
    coords_pred=loaded_dict['pre'],
    output_html='./saved_comparison.html'
)